In [1]:
import pandas as pd
from pathlib import Path
import ast

In [3]:
# 1. Setup paths and load data
data_path = Path("../Dataset")
df = pd.read_csv(data_path / "market_news_with_sentiment.csv")
stock = pd.read_csv(data_path / "stock_data.csv")

# Extract tickers from stock columns (e.g., 'adjclose_AAPL' -> 'AAPL')
tickers = [col.replace("adjclose_", "") for col in stock.columns if col.startswith("adjclose_")]

# 2. Preprocessing
df.rename(columns={"timestamp": "date"}, inplace=True)

# Map sentiment labels to numeric values
sentiment_mapping = {"positive": 1, "neutral": 0, "negative": -1}
df["sentiment_label"] = df["sentiment_label"].map(sentiment_mapping)

# Calculate signal (sentiment direction * confidence/score)
df['signal'] = df['sentiment_label'] * df['sentiment_score']

# 3. Handle the "Explode" logic
# Ensure symbols are interpreted as lists safely (handles NaNs and non-string types)
def safe_eval(val):
    if pd.isna(val): return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return [val] # Fallback if it's just a single ticker string

df["symbols"] = df["symbols"].apply(safe_eval)

# Explode symbols: one row per ticker per news item
df = df.explode("symbols")

# Filter to keep only tickers present in your stock universe
df = df[df["symbols"].isin(tickers)]

# 4. Normalize Dates
# If you want one row per CALENDAR DAY, convert to date only.
# If you want to keep specific timestamps, skip the .dt.date part.
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None).dt.date

# 5. Pivot the Table
# aggfunc='mean' handles cases where multiple news items for the same ticker happen on the same day
df_wide = df.pivot_table(
    index="date",
    columns="symbols",
    values="signal",
    aggfunc="mean"
)

# 6. Alignment
df_wide = df_wide.reindex(columns=tickers)

# Remove the name of the column index for a cleaner look
df_wide.columns.name = None

df_wide = df_wide.fillna(0)

print(df_wide.head())

                AAPL  ABBV  AMZN  AVGO  BAC  BRK-B  COST  CVX  GOOG  GOOGL  \
date                                                                         
2014-10-20  0.000000   0.0   0.0   0.0  0.0    0.0   0.0  0.0   0.0    0.0   
2014-11-07  0.000000   0.0   0.0   0.0  0.0    0.0   0.0  0.0   0.0    0.0   
2015-01-01  0.000000   0.0   0.0   0.0  0.0    0.0   0.0  0.0   0.0    0.0   
2015-01-02  0.238019   0.0   0.0   0.0  0.0    0.0   0.0  0.0   0.0    0.0   
2015-01-03  0.000000   0.0   0.0   0.0  0.0    0.0   0.0  0.0   0.0    0.0   

            ...  META  MSFT   MU  NFLX  NVDA  ORCL      TSLA    V  WMT  XOM  
date        ...                                                              
2014-10-20  ...   0.0   0.0  0.0   0.0   0.0   0.0  0.000000  0.0  0.0  0.0  
2014-11-07  ...   0.0   0.0  0.0   0.0   0.0   0.0  0.000000  0.0  0.0  0.0  
2015-01-01  ...   0.0   0.0  0.0   0.0   0.0   0.0  0.000000  0.0  0.0  0.0  
2015-01-02  ...   0.0   0.0  0.0   0.0   0.0   0.0  0.000000  0

In [4]:
df_wide.to_csv(data_path / "sentiment_signals.csv")